# 02 - Calling an LLM

Before building RAG, you need to be comfortable with its final step: sending a
prompt to an LLM and getting text back. This notebook covers the small set of
API concepts the whole course relies on.

**What you will learn**

- How a chat completion request is structured
- What the system, user, and assistant roles mean
- What temperature does
- How to run the same code against a free local model with Ollama

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")
client = OpenAI()
MODEL = "gpt-4o-mini"
print("Key loaded:", os.getenv("OPENAI_API_KEY") is not None)

Key loaded: True


## Anatomy of a request

A chat completion request has two required parts:

- **model**: which model to use. We use gpt-4o-mini because it is cheap,
  fast, and plenty good for this course.
- **messages**: the conversation so far, as a list. Each message has a
  role and content.

In [2]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Explain in two sentences what a robot is."},
    ],
)
print(response.choices[0].message.content)

A robot is a programmable machine designed to perform tasks autonomously or semi-autonomously, often mimicking human actions or functions. It typically combines hardware (such as sensors and actuators) with software to process information and execute commands.


The reply lives in response.choices[0].message.content. The response
object also carries useful metadata, for example how many tokens you used
(tokens are how usage is billed, notebook 04 explains them).

In [3]:
print("Input tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)

Input tokens:  16
Output tokens: 47


## Roles: system, user, assistant

Messages have three roles:

- **system**: instructions about how the model should behave. The model
  treats these with high priority. This is where RAG systems put rules like
  "only answer from the provided context".
- **user**: what the person asks.
- **assistant**: the model's previous replies (used to give the model memory
  of the conversation).

A system message changes behavior without changing the question:

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You answer in exactly one short sentence, no matter what."},
        {"role": "user", "content": "Explain what a robot is."},
    ],
)
print(response.choices[0].message.content)

A robot is a programmable machine designed to carry out tasks autonomously or semi-autonomously.


## Temperature

Temperature controls randomness, from 0 to 2:

- **0**: nearly deterministic, picks the most likely words. Best for factual
  tasks like RAG.
- **higher values**: more variety and creativity, but also more risk of
  drifting from the facts.

Run the next cell a few times and compare the two columns.

In [5]:
prompt = "Give me a name for a hospital delivery robot. Reply with the name only."

for temp in [0.0, 1.5]:
    names = []
    for _ in range(3):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
        )
        names.append(r.choices[0].message.content.strip())
    print(f"temperature={temp}: {names}")

temperature=0.0: ['MediBot', 'MediBot', 'MediBot']


temperature=1.5: ['MediMover', 'MediMatic', 'MedBot']


At temperature 0 the answers repeat. At 1.5 they vary. For everything
else in this course we use temperature 0, because when answering questions
from documents we want faithful, repeatable answers.

## Ollama alternative

Ollama runs open models on your own machine, free and without an API key. It
exposes the same API shape as OpenAI, so the only change is where the client
points. If you set up Ollama (see [SETUP.md](../SETUP.md)), uncomment and run:

In [6]:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# MODEL = "llama3.2"
#
# response = client.chat.completions.create(
#     model=MODEL,
#     messages=[{"role": "user", "content": "Explain in two sentences what a robot is."}],
# )
# print(response.choices[0].message.content)

Every later notebook works the same way: swap the client and model
names, and the rest of the code is unchanged.

## Exercise

1. Write a request with a system message that makes the model always answer
   in the style of a pirate, and ask it what RAG is.
2. Set temperature to 0 and run it twice. Do you get the same answer?

In [7]:
# Try the exercise here
